# Day 20 — Structured Outputs with Pydantic

**Intern:** Sehar Andleeb · **Company:** Xeven Solutions · **Repo:** `ai-internship-xeven-2026`



This notebook imports the three task scripts from `scripts/` and demonstrates each with executed output. Everything below runs **offline** (no Groq key needed); the live one-liners are at the bottom.

In [1]:
# Dependencies (install once on your machine, inside the UV env):
#   uv pip install pydantic langchain langchain-groq python-dotenv
# Notebooks never shell out to pip; this cell just confirms intent.
print("Dependencies ready.")

Dependencies ready.


## Theory

**What is Pydantic?** A data-validation library. You declare the shape you expect with Python type hints, and Pydantic validates (and where safe, coerces) incoming data at runtime, raising a clear `ValidationError` when something doesn't fit. This matters for LLM work because a model always returns *text* — even when you ask for JSON — so you need a contract between that text and your code.

**`BaseModel`.** You subclass `BaseModel` and annotate fields. Pydantic builds a validation schema at class-creation time. `model_dump()` gives a dict, `model_dump_json()` a JSON string, and `model_validate_json()` parses + validates back into an object (round-trip).

**Field types.** `str`, `int`, `float`, `bool`, `List[str]`, `Optional[...]`, `Union[...]`, plus `Field(...)` for constraints / defaults and `@field_validator` for custom single-field rules. (Pydantic v2 uses `@field_validator`, **not** v1's `@validator`.)

**LangChain integration.** `model.with_structured_output(MyModel)` binds a Pydantic schema to the chat model so `.invoke()` returns a validated object directly. `JsonOutputParser` / `PydanticOutputParser` are the fallback when a provider/model doesn't support structured output natively.

**Why structured outputs?** Reliability and type safety — you get a typed object instead of a string to regex; easier integration with downstream code; and clean error handling (retry / default on a `ValidationError`).

## Research — multi-source comparison
*(consulted 2026-06-15)*

| Topic | ChatGPT  | Gemini  | Claude | Article A (MLM) | Article B (TDS) |
|---|---|---|---|---|---|
| What is Pydantic | Runtime validation from type hints | Type-checked parsing library | Validates/coerces data at runtime, raises `ValidationError` | LLMs emit text, not data; Pydantic enforces the schema | Validates input field types, limits, custom constraints |
| BaseModel | Subclass + annotate fields | Schema built at class creation | Subclass, annotate, `model_dump`/`model_validate_json` round-trip | All models inherit `BaseModel` for auto-validation | Define models, fields validated in declaration order |
| Field types | str/int/List/Optional/Union + validators | Same set + `Field()` constraints | Type hints + `Field()` + `@field_validator`; `Optional[x]=None` ⇒ optional | `Optional[str]=None` may be missing/null; `@field_validator` for custom logic | `gt/ge/lt/le`, `min_length`/`pattern`; before/after/plain/wrap modes |
| LangChain integration | `with_structured_output()` preferred | Structured output or `JsonOutputParser` | `model.with_structured_output(M)`; parser is the fallback | Parse JSON then validate; retry on bad output | (validator mechanics focus; not LangChain-specific) |
| Why structured outputs | Reliability + type safety | Fewer parse errors | Typed objects, clean error handling, easy integration | Catches missing/extra/mistyped fields before they crash code | Enforces constraints, auto-converts in non-strict mode |

**Clearest explanation (Claude):** An LLM always hands you a string; Pydantic is the gate that turns that string into a typed, trusted object — or tells you exactly why it can't.

**Article A:** *The Complete Guide to Using Pydantic for Validating LLM Outputs* — MachineLearningMastery (Dec 2025): https://machinelearningmastery.com/the-complete-guide-to-using-pydantic-for-validating-llm-outputs/

**Article B:** *Validations in Pydantic V2* — Towards Data Science: https://towardsdatascience.com/validations-in-pydantic-v2-15bcbb39a98b/

## Task 1 — Pydantic Model Suite
`Person` (name, age, email, hobbies) and `Product` (id, name, price, in_stock, tags) with validators: **email format**, **age > 0**, **price ≥ 0**. Valid data constructs; each invalid case raises `ValidationError`.

In [2]:
import sys
sys.path.insert(0, "scripts")
import task1_pydantic_models as t1

print('case'.ljust(42), 'expected'.ljust(9),
      'outcome'.ljust(16), 'ok')
print("-" * 78)
for label, builder, should in t1.build_cases():
    r = t1.run_case(label, builder, should)
    print(f"{r['case']:<42} {r['expected']:<9} "
          f"{r['outcome']:<16} {r['behaved_as_expected']}")

case                                       expected  outcome          ok
------------------------------------------------------------------------------
Person: valid                              pass      constructed      True
Person: bad email format                   fail      ValidationError  True
Person: age == 0                           fail      ValidationError  True
Person: negative age                       fail      ValidationError  True
Product: valid                             pass      constructed      True
Product: price == 0 (boundary, allowed)    pass      constructed      True
Product: negative price                    fail      ValidationError  True


In [3]:
# A single invalid Person surfaces BOTH failing validators at once.
from pydantic import ValidationError
try:
    t1.Person(name="X", age=-1, email="not-an-email")
except ValidationError as exc:
    for err in exc.errors():
        print(err['loc'], '->', err['msg'])

('age',) -> Value error, age must be greater than 0
('email',) -> Value error, email is not a valid address


In [4]:
# JSON export + re-load round-trip.
p = t1.Person(name="Sehar", age=22, email="sehar@example.com",
              hobbies=["reading", "coding"])
as_json = p.model_dump_json()
restored = t1.Person.model_validate_json(as_json)
print("JSON   :", as_json)
print("Equal? :", restored == p)

JSON   : {"name":"Sehar","age":22,"email":"sehar@example.com","hobbies":["reading","coding"]}
Equal? : True


## Task 2 — LLM → Structured Data Pipeline
`Article` (title, author, published_date, summary, tags). A batch of 12 auto-created articles is run through the pipeline with **retry on validation failure** and **default-value fallback**. Article 10 is rigged to fail once then succeed (retry); article 11 always fails (fallback).

In [5]:
import task2_structured_pipeline as t2

samples = t2.build_sample_articles()
dataset, via_retry, defaults = [], 0, 0
for s in samples:
    res = t2.extract_one(s, None)  # None => offline stub
    if res['used_default']:
        defaults += 1
    elif res['attempts'] > 1:
        via_retry += 1
    dataset.append(res)

ok = len(samples) - defaults
print(f"processed            : {len(samples)}")
print(f"extracted ok         : {ok}")
print(f"recovered via retry  : {via_retry}")
print(f"fell back to default : {defaults}")
print(f"approx input tokens  : ~{t2.estimate_tokens(samples)} (cost ~$0.00, free tier)")

processed            : 12
extracted ok         : 11
recovered via retry  : 1
fell back to default : 1
approx input tokens  : ~336 (cost ~$0.00, free tier)


In [6]:
# One clean Article, the retried one, and the defaulted one.
import json
print("clean   :", json.dumps(dataset[0]["article"].model_dump()))
print("retried :", "attempts =", dataset[10]["attempts"],
      dataset[10]["article"].title)
print("default :", json.dumps(dataset[11]["article"].model_dump()))

clean   : {"title": "Groq Posts Record Low-Latency Inference", "author": "A. Khan", "published_date": "2026-02-10", "summary": "Groq reported a new throughput record for its LPU stack.", "tags": ["hardware", "inference", "llm"]}
retried : attempts = 2 Prompt Templating Best Practice
default : {"title": "UNKNOWN", "author": "UNKNOWN", "published_date": "1970-01-01", "summary": "", "tags": []}


## Task 3 — Multi-Entity Extraction System
Nested models `Company` → `Employee[]`. Extract from auto-created press releases, build a knowledge graph (company → employee → skills), validate, and **score against gold labels**. The offline stub injects three documented errors, so accuracy is realistic (not a trivial 100%).

In [7]:
import task3_entity_extraction as t3

companies = [t3.Company(**t3.offline_extract(g)) for g in t3.GOLD]
graph = t3.to_graph(companies)
metrics = t3.score(companies)

print(f"companies : {len(companies)}")
print(f"employees : {sum(len(c.employees) for c in companies)}")
print(f"field accuracy : {metrics['field_accuracy']*100:.1f}% "
      f"({metrics['correct_facts']}/{metrics['total_facts']})")
print(f"skill P/R/F1   : {metrics['skill_precision']:.2f} / "
      f"{metrics['skill_recall']:.2f} / {metrics['skill_f1']:.2f}")

companies : 3
employees : 5
field accuracy : 92.0% (23/25)
skill P/R/F1   : 0.93 / 0.93 / 0.93


In [8]:
import json
# Knowledge-graph slice for one company.
print(json.dumps(graph["NimbusAI"], indent=2))

{
  "location": "Lahore, PK",
  "employees": {
    "Ayesha Tariq": {
      "title": "ML Engineer",
      "department": "Research",
      "skills": [
        "pytorch",
        "nlp"
      ]
    },
    "Bilal Sheikh": {
      "title": "Data Engineer",
      "department": "Platform",
      "skills": [
        "spark",
        "airflow",
        "sql",
        "kafka"
      ]
    }
  }
}


## Offline vs live — what the numbers mean

Every demo above runs the **offline stub** on purpose, for two reasons:

1. **Reproducible** — it runs with no API key, so this notebook always executes the same way for anyone.
2. **It exercises the failure paths** — the stub deliberately feeds one broken article (to trigger a **retry**), one always-broken article (to trigger the **default fallback**), and three small extraction errors in Task 3 (so the **accuracy metric** runs on imperfect data). Clean live data wouldn't trigger any of these, so you'd never *see* the safeguards working.

On a **live** Groq run with this clean sample data, the model parses everything correctly, so the numbers are different (and better):

| Task | Offline (stub, above) | Live (Groq, clean data) |
|---|---|---|
| Task 2 | 11/12 ok, 1 retry, 1 default | 12/12 ok, 0 retry, 0 default |
| Task 3 | 92.0% (23/25), F1 0.93 | 100% (25/25), F1 1.00 |

The **live** numbers are the ones in `scripts/outputs/` and in the progress email. The cell below reproduces them if a `GROQ_API_KEY` is set.

In [9]:
# Optional live verification. Runs only if GROQ_API_KEY is set,
# so the notebook still executes offline without it.
import os

if os.getenv("GROQ_API_KEY"):
    ex2 = t2.build_live_extractor()
    live2 = [t2.extract_one(s, ex2) for s in samples]
    ok = sum(0 if r['used_default'] else 1 for r in live2)
    print(f"[live] task2 extracted ok : {ok}/{len(samples)}")

    ex3 = t3.build_live_extractor()
    comps = [t3.Company(**ex3(r)) for r in t3.build_press_releases()]
    m = t3.score(comps)
    print(f"[live] task3 field accuracy: "
          f"{m['field_accuracy'] * 100:.1f}% "
          f"({m['correct_facts']}/{m['total_facts']})")
else:
    print("No GROQ_API_KEY set - skipping live cell "
          "(offline demos above stand on their own).")

c:\Users\PMLS\Desktop\ai-internship-xeven-2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[live] task2 extracted ok : 12/12
[live] task3 field accuracy: 100.0% (25/25)


## Running the real thing (live Groq)
Everything above used the offline stub so the notebook is reproducible. On your machine, with `GROQ_API_KEY` in a local `.env`, run the full live pipeline from inside `day20/scripts/`:

```bash
uv run python task1_pydantic_models.py        # pure Pydantic, no API
uv run python task2_structured_pipeline.py --live
uv run python task3_entity_extraction.py --live
```

The `--live` flag swaps the stub for `ChatGroq(model="llama-3.3-70b-versatile").with_structured_output(...)`. The key is read via `os.getenv("GROQ_API_KEY")` after `load_dotenv()` — it never appears in any `.py` or `.ipynb`.